In [ ]:
%pip install -U chromadb ollama

In [ ]:
from pathlib import Path
import sqlite3
import json
import re
from datetime import datetime, timezone

import numpy as np
import pandas as pd

import chromadb
import ollama

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.max_colwidth",
    180
)

In [ ]:
def find_project_root(start_path: Path) -> Path:

    start_path = start_path.resolve()

    for candidate in [
        start_path,
        *start_path.parents
    ]:

        database_path = (
            candidate
            / "database"
            / "customer_churn.db"
        )

        if database_path.exists():

            return candidate

    raise FileNotFoundError(
        "Could not locate "
        "database/customer_churn.db"
    )


project_root = find_project_root(
    Path.cwd()
)


database_path = (
    project_root
    / "database"
    / "customer_churn.db"
)


rag_directory = (
    project_root
    / "rag"
)


knowledge_base_directory = (
    rag_directory
    / "knowledge_base"
)


vector_database_directory = (
    rag_directory
    / "chroma_db"
)


rag_output_directory = (
    project_root
    / "reports"
    / "rag"
)


for directory in [
    rag_directory,
    knowledge_base_directory,
    vector_database_directory,
    rag_output_directory
]:

    directory.mkdir(
        parents=True,
        exist_ok=True
    )


print(
    "Project root:",
    project_root
)

print(
    "Database:",
    database_path
)

In [ ]:
retention_policies = [

    {
        "policy_id": "RET-001",

        "title": (
            "Month-to-Month Contract "
            "Conversion Offer"
        ),

        "category": "Contract",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers currently using a month-to-month
contract may be offered an incentive to migrate
to a one-year contract.

Recommended action:
Offer a one-year contract with a temporary
service discount or loyalty credit.

Suitable when:
- Contract is Month-to-month.
- Churn risk is medium or high.
- Customer has not already committed to a
  long-term contract.

Objective:
Improve retention by increasing contractual
commitment.

Do not recommend this policy when the customer
already has a one-year or two-year contract.
"""
    },


    {
        "policy_id": "RET-002",

        "title": (
            "New Customer Early-Life "
            "Retention Program"
        ),

        "category": "Tenure",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers with tenure of six months or less
may be enrolled in an early-life retention
program.

Recommended actions may include:
- Welcome loyalty credit.
- Proactive service check-in.
- Plan review.
- Setup or onboarding assistance.

Suitable when:
- Tenure is 0 to 6 months.
- Customer shows elevated churn risk.

Objective:
Address dissatisfaction during the customer's
early relationship with the company.
"""
    },


    {
        "policy_id": "RET-003",

        "title": (
            "Online Security Trial"
        ),

        "category": "Service",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

An internet customer without online security
may receive a temporary online-security trial.

Recommended action:
Offer online security at no additional charge
for three months, followed by normal pricing.

Suitable when:
- Customer has internet service.
- Online security is No.
- Churn risk is elevated.

Do not apply when online security is already
included or internet service is unavailable.
"""
    },


    {
        "policy_id": "RET-004",

        "title": (
            "Technical Support Retention Offer"
        ),

        "category": "Support",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers with internet service but without
technical support may be offered temporary
premium technical support.

Recommended action:
Provide three months of technical-support
service or a proactive technical health check.

Suitable when:
- Tech support is No.
- Customer has internet service.
- Customer is in a churn-risk segment.

Objective:
Reduce service-related dissatisfaction and
increase perceived support quality.
"""
    },


    {
        "policy_id": "RET-005",

        "title": (
            "Electronic Payment Migration"
        ),

        "category": "Payment",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers using electronic check may be
encouraged to move to an automated electronic
payment method where available.

Recommended action:
Explain automatic payment options and provide
a small one-time account credit for successful
migration.

Suitable when:
- Payment method is Electronic check.
- Customer is campaign eligible.

The recommendation should be presented as an
optional convenience improvement.
"""
    },


    {
        "policy_id": "RET-006",

        "title": (
            "High-CLTV Priority Retention"
        ),

        "category": "Value",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers with high customer lifetime value
and elevated churn risk should receive priority
retention attention.

Recommended actions:
- Prioritized human outreach.
- Personalized plan review.
- Loyalty credit where appropriate.
- Escalation to a retention specialist.

Suitable when:
- Customer has high CLTV relative to the
  customer population.
- Customer is campaign eligible or high risk.

Objective:
Prioritize retention resources according to
both churn risk and customer value.
"""
    },


    {
        "policy_id": "RET-007",

        "title": (
            "Fiber Service Experience Review"
        ),

        "category": "Internet",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

High-risk fiber-optic customers may receive a
proactive service-experience review.

Recommended actions:
- Review recent service experience.
- Verify service configuration.
- Offer technical assistance where needed.
- Review whether the current plan matches the
  customer's requirements.

Suitable when:
- Internet service is Fiber optic.
- Churn probability is elevated.

Do not assume that fiber service itself caused
the customer's churn risk.
"""
    },


    {
        "policy_id": "RET-008",

        "title": (
            "High Monthly Charge Plan Review"
        ),

        "category": "Billing",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers with relatively high monthly charges
and elevated churn risk may receive a plan
optimization review.

Recommended action:
Compare the customer's active services with
available plan configurations and identify
unnecessary or poorly matched services.

Suitable when:
- Monthly charge is high relative to the
  customer population.
- Customer is campaign eligible.

The system must not invent a discount amount
that is not present in the policy knowledge
base.
"""
    },


    {
        "policy_id": "RET-009",

        "title": (
            "Long-Term Contract Loyalty Program"
        ),

        "category": "Loyalty",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Customers already on one-year or two-year
contracts should not be recommended the
month-to-month conversion offer.

For elevated-risk long-term customers,
recommended actions include:
- Loyalty acknowledgement.
- Service review.
- Support check-in.
- Plan optimization.

Suitable when:
- Contract is One year or Two year.
- Customer has elevated churn risk.
"""
    },


    {
        "policy_id": "RET-010",

        "title": (
            "Low-Risk Customer Handling"
        ),

        "category": "Campaign",

        "content": """
PROJECT DEMONSTRATION POLICY — SYNTHETIC.

Low-risk customers should generally not receive
costly retention incentives solely because a
prediction system exists.

Recommended action:
Continue standard engagement and customer
service.

Suitable when:
- Customer is classified as Low Risk.
- Customer is not campaign eligible.

Objective:
Avoid unnecessary retention spending.
"""
    }
]

In [ ]:
policy_df = pd.DataFrame(
    retention_policies
)


print(
    "Number of policies:",
    len(policy_df)
)


display(
    policy_df[
        [
            "policy_id",
            "title",
            "category"
        ]
    ]
)

In [ ]:
knowledge_base_path = (
    knowledge_base_directory
    / "retention_policies.csv"
)


policy_df.to_csv(
    knowledge_base_path,
    index=False
)


print(
    "Knowledge base saved to:"
)

print(
    knowledge_base_path
)

In [ ]:
chroma_client = chromadb.PersistentClient(
    path=str(
        vector_database_directory
    )
)


COLLECTION_NAME = (
    "retention_policies"
)
try:

    chroma_client.delete_collection(
        name=COLLECTION_NAME
    )

    print(
        "Old collection removed."
    )

except Exception:

    pass
policy_collection = (
    chroma_client.create_collection(
        name=COLLECTION_NAME,

        metadata={
            "description":
                "Synthetic telecom retention "
                "policies for churn project"
        }
    )
)


print(
    "Collection created:",
    COLLECTION_NAME
)

In [ ]:
policy_collection.add(

    ids=policy_df[
        "policy_id"
    ].tolist(),

    documents=policy_df[
        "content"
    ].tolist(),

    metadatas=[
        {
            "policy_id": row[
                "policy_id"
            ],

            "title": row[
                "title"
            ],

            "category": row[
                "category"
            ]
        }

        for _, row
        in policy_df.iterrows()
    ]
)


print(
    "Documents stored:",
    policy_collection.count()
)

In [ ]:
test_query = """
High-risk new customer with one month tenure,
month-to-month contract and no online security.
"""


test_results = (
    policy_collection.query(

        query_texts=[
            test_query
        ],

        n_results=4
    )
)
retrieval_test_df = pd.DataFrame(
    {
        "policy_id": [
            metadata[
                "policy_id"
            ]

            for metadata
            in test_results[
                "metadatas"
            ][0]
        ],

        "title": [
            metadata[
                "title"
            ]

            for metadata
            in test_results[
                "metadatas"
            ][0]
        ],

        "distance": (
            test_results[
                "distances"
            ][0]
        )
    }
)


display(
    retrieval_test_df
)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    sample_customers = pd.read_sql_query(
        """
        SELECT *

        FROM vw_customer_risk

        WHERE
            churn_probability IS NOT NULL

        ORDER BY
            churn_probability DESC

        LIMIT 10;
        """,

        connection
    )


display(
    sample_customers
)

In [ ]:
with sqlite3.connect(
    database_path
) as connection:

    explained_customers = (
        pd.read_sql_query(
            """
            SELECT
                *

            FROM vw_customer_risk

            WHERE
                top_risk_reasons IS NOT NULL
                AND churn_probability IS NOT NULL

            ORDER BY
                churn_probability DESC;
            """,

            connection
        )
    )


print(
    "Customers with explanations:",
    len(explained_customers)
)


display(
    explained_customers[
        [
            "customerid",
            "churn_probability",
            "business_risk_segment",
            "cltv",
            "contract",
            "tenure_months",
            "top_risk_reasons"
        ]
    ].head(10)
)

In [ ]:
def get_customer_profile(
    customer_id
):

    query = """
    SELECT *

    FROM vw_customer_risk

    WHERE customerid = ?;
    """


    with sqlite3.connect(
        database_path
    ) as connection:

        customer_df = pd.read_sql_query(
            query,

            connection,

            params=[
                customer_id
            ]
        )


    if customer_df.empty:

        raise ValueError(
            f"Customer not found: "
            f"{customer_id}"
        )


    return customer_df.iloc[0]

In [ ]:
demo_customer_id = (
    explained_customers
    .iloc[0][
        "customerid"
    ]
)


demo_customer = (
    get_customer_profile(
        demo_customer_id
    )
)


print(
    "Selected customer:",
    demo_customer_id
)


display(
    demo_customer.to_frame(
        name="value"
    )
)

In [ ]:
def clean_value(value):

    if pd.isna(value):

        return "Unknown"

    return str(value)


def build_retrieval_query(
    customer
):

    query = f"""
Telecom customer retention recommendation.

Risk segment:
{clean_value(
    customer.get(
        "business_risk_segment"
    )
)}

Churn probability:
{clean_value(
    customer.get(
        "churn_probability"
    )
)}

Tenure:
{clean_value(
    customer.get(
        "tenure_months"
    )
)} months

Contract:
{clean_value(
    customer.get(
        "contract"
    )
)}

Internet service:
{clean_value(
    customer.get(
        "internet_service"
    )
)}

Monthly charges:
{clean_value(
    customer.get(
        "monthly_charges"
    )
)}

CLTV:
{clean_value(
    customer.get(
        "cltv"
    )
)}

Top churn-risk reasons:
{clean_value(
    customer.get(
        "top_risk_reasons"
    )
)}
"""

    return query.strip()

In [ ]:
demo_retrieval_query = (
    build_retrieval_query(
        demo_customer
    )
)


print(
    demo_retrieval_query
)

In [ ]:
def retrieve_policies(
    customer,
    top_k=4
):

    retrieval_query = (
        build_retrieval_query(
            customer
        )
    )


    results = (
        policy_collection.query(

            query_texts=[
                retrieval_query
            ],

            n_results=top_k
        )
    )


    retrieved_policies = []


    for position in range(
        len(
            results[
                "ids"
            ][0]
        )
    ):

        retrieved_policies.append(
            {
                "policy_id":
                    results[
                        "ids"
                    ][0][
                        position
                    ],

                "document":
                    results[
                        "documents"
                    ][0][
                        position
                    ],

                "metadata":
                    results[
                        "metadatas"
                    ][0][
                        position
                    ],

                "distance":
                    results[
                        "distances"
                    ][0][
                        position
                    ]
            }
        )


    return (
        retrieval_query,
        retrieved_policies
    )

In [ ]:
(
    demo_retrieval_query,
    demo_retrieved_policies
) = retrieve_policies(
    demo_customer,
    top_k=4
)
retrieved_policy_table = pd.DataFrame(
    [
        {
            "rank": index + 1,

            "policy_id":
                policy[
                    "policy_id"
                ],

            "title":
                policy[
                    "metadata"
                ][
                    "title"
                ],

            "category":
                policy[
                    "metadata"
                ][
                    "category"
                ],

            "distance":
                policy[
                    "distance"
                ]
        }

        for index, policy
        in enumerate(
            demo_retrieved_policies
        )
    ]
)


display(
    retrieved_policy_table
)

Check local LLM

In [ ]:
GENERATION_MODEL = (
    "qwen3:4b-instruct"
)


try:

    test_response = ollama.chat(

        model=GENERATION_MODEL,

        messages=[
            {
                "role": "user",
                "content":
                    "Reply only with: "
                    "Ollama connection successful."
            }
        ]
    )


    print(
        test_response[
            "message"
        ][
            "content"
        ]
    )


except Exception as error:

    print(
        "Could not connect to Ollama."
    )

    print(
        "\nMake sure Ollama is running "
        "and the model has been pulled."
    )

    print(
        "\nError:"
    )

    print(error)

In [ ]:
def format_policy_context(
    retrieved_policies
):

    context_sections = []


    for rank, policy in enumerate(
        retrieved_policies,
        start=1
    ):

        metadata = (
            policy[
                "metadata"
            ]
        )


        section = f"""
SOURCE {rank}
Policy ID: {policy['policy_id']}
Title: {metadata['title']}
Category: {metadata['category']}

{policy['document']}
"""

        context_sections.append(
            section.strip()
        )


    return "\n\n---\n\n".join(
        context_sections
    )

In [ ]:
def format_customer_context(
    customer
):

    fields = [
        "customerid",
        "churn_probability",
        "business_risk_segment",
        "campaign_eligible",
        "high_priority_customer",
        "campaign_rank",
        "cltv",
        "priority_score",
        "tenure_months",
        "contract",
        "payment_method",
        "monthly_charges",
        "total_charges",
        "gender",
        "senior_citizen",
        "partner",
        "dependents",
        "top_risk_reasons"
    ]


    lines = []


    for field in fields:

        if field in customer.index:

            lines.append(
                f"{field}: "
                f"{clean_value(customer[field])}"
            )


    return "\n".join(lines)

In [ ]:
RAG_SYSTEM_PROMPT = """
You are a telecom customer-retention assistant.

Your job is to recommend retention actions for
a customer using ONLY:

1. The structured customer information provided.
2. The retrieved retention-policy documents.

Rules:

- Do not calculate or change the churn probability.
  The probability was produced by the ML model.

- Treat SHAP reasons as predictive model drivers,
  not proof that a feature caused churn.

- Do not invent discounts, durations, products,
  eligibility rules, or policy conditions.

- Recommend only actions supported by the
  retrieved policy documents.

- Every recommended action must cite at least
  one Policy ID such as [RET-001].

- If the retrieved policies do not support a
  requested recommendation, explicitly say that
  the knowledge base does not provide sufficient
  information.

- Prefer practical, concise recommendations.

Return your answer using exactly these sections:

Risk Summary:
...

Recommended Actions:
1. ...
2. ...
3. ...

Why These Actions:
...

Policy Sources:
- RET-...
- RET-...
"""

In [ ]:
def build_rag_prompt(
    customer,
    retrieved_policies
):

    customer_context = (
        format_customer_context(
            customer
        )
    )


    policy_context = (
        format_policy_context(
            retrieved_policies
        )
    )


    user_prompt = f"""
CUSTOMER INFORMATION
--------------------
{customer_context}


RETRIEVED RETENTION POLICIES
----------------------------
{policy_context}


TASK
----
Recommend appropriate retention actions for
this customer.

Do not recommend anything that is unsupported
by the retrieved policies.
"""


    return user_prompt.strip()

In [ ]:
demo_rag_prompt = (
    build_rag_prompt(
        demo_customer,
        demo_retrieved_policies
    )
)


print(
    demo_rag_prompt
)

In [ ]:
def generate_retention_recommendation(
    customer_id,
    top_k=4
):

    customer = (
        get_customer_profile(
            customer_id
        )
    )


    (
        retrieval_query,
        retrieved_policies
    ) = retrieve_policies(
        customer,
        top_k=top_k
    )


    rag_prompt = (
        build_rag_prompt(
            customer,
            retrieved_policies
        )
    )


    response = ollama.chat(

        model=GENERATION_MODEL,

        messages=[
            {
                "role": "system",
                "content":
                    RAG_SYSTEM_PROMPT
            },

            {
                "role": "user",
                "content":
                    rag_prompt
            }
        ],

        options={
            "temperature": 0.2
        }
    )


    answer = (
        response[
            "message"
        ][
            "content"
        ]
    )


    return {
        "customer":
            customer,

        "retrieval_query":
            retrieval_query,

        "retrieved_policies":
            retrieved_policies,

        "answer":
            answer
    }

In [ ]:
demo_rag_result = (
    generate_retention_recommendation(
        customer_id=demo_customer_id,
        top_k=4
    )
)


print(
    demo_rag_result[
        "answer"
    ]
)

In [ ]:
def extract_policy_ids(
    text
):

    return sorted(
        set(
            re.findall(
                r"RET-\d{3}",
                text
            )
        )
    )


used_policy_ids = (
    extract_policy_ids(
        demo_rag_result[
            "answer"
        ]
    )
)


retrieved_policy_ids = [
    policy[
        "policy_id"
    ]

    for policy
    in demo_rag_result[
        "retrieved_policies"
    ]
]


print(
    "Policies cited by LLM:",
    used_policy_ids
)

print(
    "Policies retrieved:",
    retrieved_policy_ids
)

In [ ]:
unsupported_policy_ids = [
    policy_id

    for policy_id
    in used_policy_ids

    if policy_id
    not in retrieved_policy_ids
]


if unsupported_policy_ids:

    print(
        "WARNING:"
    )

    print(
        "LLM cited policies that were "
        "not retrieved:"
    )

    print(
        unsupported_policy_ids
    )

else:

    print(
        "Citation grounding check passed."
    )

In [ ]:
rag_test_customers = (
    explained_customers
    .sort_values(
        "churn_probability",
        ascending=False
    )
    .groupby(
        "business_risk_segment",
        group_keys=False
    )
    .head(2)
)


display(
    rag_test_customers[
        [
            "customerid",
            "churn_probability",
            "business_risk_segment",
            "contract",
            "tenure_months",
            "cltv"
        ]
    ]
)


In [ ]:
rag_test_results = []


for customer_id in (
    rag_test_customers[
        "customerid"
    ]
):

    print(
        "=" * 70
    )

    print(
        "Customer:",
        customer_id
    )


    result = (
        generate_retention_recommendation(
            customer_id,
            top_k=4
        )
    )


    print(
        result[
            "answer"
        ]
    )


    retrieved_ids = [
        policy[
            "policy_id"
        ]

        for policy
        in result[
            "retrieved_policies"
        ]
    ]


    cited_ids = (
        extract_policy_ids(
            result[
                "answer"
            ]
        )
    )


    rag_test_results.append(
        {
            "customerid":
                customer_id,

            "churn_probability":
                float(
                    result[
                        "customer"
                    ][
                        "churn_probability"
                    ]
                ),

            "business_risk_segment":
                result[
                    "customer"
                ][
                    "business_risk_segment"
                ],

            "retrieved_policy_ids":
                ", ".join(
                    retrieved_ids
                ),

            "cited_policy_ids":
                ", ".join(
                    cited_ids
                ),

            "grounding_passed":
                set(
                    cited_ids
                ).issubset(
                    set(
                        retrieved_ids
                    )
                ),

            "recommendation":
                result[
                    "answer"
                ]
        }
    )

In [ ]:
rag_evaluation_df = pd.DataFrame(
    rag_test_results
)


display(
    rag_evaluation_df[
        [
            "customerid",
            "churn_probability",
            "business_risk_segment",
            "retrieved_policy_ids",
            "cited_policy_ids",
            "grounding_passed"
        ]
    ]
)
grounding_pass_rate = (
    rag_evaluation_df[
        "grounding_passed"
    ].mean()
)


print(
    "Citation grounding pass rate:",
    f"{grounding_pass_rate:.1%}"
)

In [ ]:
rag_evaluation_path = (
    rag_output_directory
    / "rag_recommendation_evaluation.csv"
)


rag_evaluation_df.to_csv(
    rag_evaluation_path,
    index=False
)


print(
    "RAG evaluation saved to:"
)

print(
    rag_evaluation_path
)

In [ ]:
rag_metadata = {

    "notebook":
        "08_rag_retention_recommendation",

    "architecture":
        (
            "SQL structured customer context "
            "+ Chroma policy retrieval "
            "+ Ollama LLM generation"
        ),

    "generation_model":
        GENERATION_MODEL,

    "vector_database":
        "Chroma",

    "embedding_model":
        (
            "Chroma default "
            "all-MiniLM-L6-v2"
        ),

    "knowledge_base":
        (
            "Synthetic project demonstration "
            "telecom retention policies"
        ),

    "policy_count":
        int(
            policy_collection.count()
        ),

    "retrieval_top_k":
        4,

    "grounding_validation":
        (
            "LLM policy citations checked "
            "against retrieved policy IDs"
        ),

    "created_at_utc":
        datetime.now(
            timezone.utc
        ).isoformat()
}


rag_metadata_path = (
    rag_output_directory
    / "rag_metadata.json"
)


with open(
    rag_metadata_path,
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        rag_metadata,
        file,
        indent=4
    )


print(
    "Metadata saved to:"
)

print(
    rag_metadata_path
)

In [ ]:
print(
    "=" * 70
)

print(
    "NOTEBOOK 8 — RAG RETENTION SYSTEM COMPLETE"
)

print(
    "=" * 70
)

print(
    "Structured source:"
)

print(
    "SQLite customer churn database"
)

print(
    "\nUnstructured knowledge source:"
)

print(
    "Synthetic retention-policy catalogue"
)

print(
    "\nEmbedding/vector layer:"
)

print(
    "Chroma + all-MiniLM-L6-v2"
)

print(
    "\nGeneration model:"
)

print(
    GENERATION_MODEL
)

print(
    "\nRAG workflow:"
)

print(
    "Customer profile"
)

print(
    "→ retrieval query"
)

print(
    "→ vector search"
)

print(
    "→ top policy documents"
)

print(
    "→ grounded LLM prompt"
)

print(
    "→ retention recommendation"
)

print(
    "→ policy citation validation"
)